# Gated PixelCNN demo

Class-conditional Gated PixelCNN (van den Oord et al., 2016) on binarized MNIST.
Train first, outside the notebook:

```bash
uv run python -m pixelcnn_demo.train
```

The widget below runs the trained model in the browser: blind spot, sampling, inpainting.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import torch

from pixelcnn_demo import PixelCNNDemo, load_run

RUN = Path("runs/gated")

In [ ]:
demo = PixelCNNDemo.from_run(RUN)
demo  # demo.tab = "sample" switches tabs from Python

## Training curves

Negative log-likelihood in nats per image (784 binary pixels). Divide by 784 · ln 2 for bits/dim.

In [ ]:
model, metrics, history = load_run(RUN)
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(figsize=(6, 3.5))
for key, label in [("train", "train (10% no class)"), ("test", "test, true class"), ("test_uncond", "test, no class")]:
    ax.plot(ep, [h[key] for h in history], label=label)
ax.set_xlabel("epoch"); ax.set_ylabel("−log p(x) (nats)"); ax.legend(); ax.grid(alpha=0.3)
metrics

## Samples in PyTorch

The naive way: one full forward pass per pixel, 784 in all, for a whole batch at once.

In [ ]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(dev)
y = torch.arange(11).repeat_interleave(8)  # 10 = no class
x = model.sample(y, temperature=1.0, generator=torch.Generator(dev).manual_seed(0)).cpu()
grid = x.reshape(11, 8, 28, 28).permute(0, 2, 1, 3).reshape(11 * 28, 8 * 28)
plt.figure(figsize=(4, 5.5)); plt.imshow(grid, cmap="gray")
plt.yticks([28 * i + 14 for i in range(11)], [*map(str, range(10)), "none"]); plt.xticks([]);

## Blind spot, from gradients

Which input pixels move the output at one position? Nonzero gradients of random-weight networks give the receptive field; the demo's first tab draws the same sets from the architecture.

In [ ]:
from pixelcnn_demo import GatedPixelCNN, OriginalPixelCNN

def rf(m, r=10, c=8, n=16):
    x = torch.rand(1, 1, n, n, dtype=torch.float64, requires_grad=True)
    m.double().features(x, torch.tensor([0]))[0, :, r, c].sum().backward()
    g = (x.grad[0, 0] != 0).float(); g[r, c] = 0.5
    return g

fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
for ax, (name, m) in zip(axes, [("PixelCNN, 6 layers", OriginalPixelCNN(layers=6)),
                               ("Gated PixelCNN, 6 layers", GatedPixelCNN(channels=4, layers=6, k_first=3, k=3, head=4))]):
    ax.imshow(rf(m), cmap="Blues", vmin=0, vmax=1); ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])